In [0]:
from pyspark.sql import DataFrame
from pyspark.sql.functions import col, count, when, isnan, isnull, trim, length, current_timestamp, from_utc_timestamp, row_number, udf, lit
from pyspark.sql.types import StringType, BooleanType
from pyspark.sql.window import Window

# Configuração
CATALOG = "workspace"
SCHEMA_BRONZE = "yelp_ing"
SCHEMA_SILVER = "yelp_ing"  # Ajustar se necessário

# ========== FILTRO DE RESTAURANTES E ESTABELECIMENTOS DE COMIDA ==========

FOOD_TAGS = {
    "Restaurants", "Food", "Fast Food", "Food Trucks", "Food Delivery Services",
    "Pizza", "Mexican", "Chinese", "Italian", "Japanese", "Thai", "Vietnamese",
    "Indian", "Greek", "Mediterranean", "French", "Korean", "Filipino", "African",
    "Cuban", "Caribbean", "Middle Eastern", "Latin American", "Asian Fusion",
    "American (Traditional)", "American (New)", "Canadian (New)", "Cajun/Creole",
    "Pakistani", "Southern", "Soul Food", "Tex-Mex",
    "Burgers", "Sandwiches", "Sushi Bars", "Seafood", "Steakhouses", "Barbeque",
    "Chicken Wings", "Chicken Shop", "Hot Dogs", "Cheesesteaks", "Tacos",
    "Delis", "Diners", "Buffets", "Breakfast & Brunch",
    "Cafes", "Coffee & Tea", "Coffee Roasteries", "Tea Rooms", "Bubble Tea",
    "Juice Bars & Smoothies", "Desserts", "Bakeries", "Donuts", "Bagels",
    "Ice Cream & Frozen Yogurt", "Candy Stores",
    "Bars", "Nightlife", "Breweries", "Wine & Spirits", "Beer", "Cocktail Bars",
    "Dive Bars", "Sports Bars", "Pubs", "Gastropubs", "Lounges",
    "Grocery", "Specialty Food", "Seafood Markets", "Meat Shops", "Fruits & Veggies",
    "Farmers Market", "Convenience Stores", "Wholesale Stores",
    "Caterers",
}

def is_food_related(categories_str):
    """Verifica se o estabelecimento é relacionado a alimentação."""
    if not categories_str:
        return False
    tags = {t.strip() for t in str(categories_str).split(",")}
    return bool(tags & FOOD_TAGS)

def get_segmento_alimentacao(categories_str):
    """Retorna o segmento de alimentação baseado nas categorias."""
    if not categories_str:
        return "Outros alimentação"
    
    tags = {t.strip() for t in str(categories_str).split(",")}
    
    if "Restaurants" in tags:
        return "RESTAURANTE"
    if tags & {"Bars", "Nightlife", "Breweries", "Cocktail Bars", "Dive Bars",
                "Sports Bars", "Pubs", "Gastropubs", "Lounges", "Wine & Spirits", "Beer"}:
        return "BAR E BEBIDA"
    if tags & {"Coffee & Tea", "Coffee Roasteries", "Tea Rooms", "Bubble Tea",
                "Cafes", "Juice Bars & Smoothies"}:
        return "CAFE"
    if tags & {"Bakeries", "Donuts", "Bagels", "Ice Cream & Frozen Yogurt",
                "Candy Stores", "Desserts"}:
        return "PADARIA"
    segmentos_alimentacao = {
        "Restaurants",
        "Bars", "Nightlife", "Breweries", "Cocktail Bars", "Dive Bars",
        "Sports Bars", "Pubs", "Gastropubs", "Lounges", "Wine & Spirits", "Beer",
        "Coffee & Tea", "Coffee Roasteries", "Tea Rooms", "Bubble Tea",
        "Cafes", "Juice Bars & Smoothies",
        "Bakeries", "Donuts", "Bagels", "Ice Cream & Frozen Yogurt",
        "Candy Stores", "Desserts"
    }
    if not tags & segmentos_alimentacao:
        return "OUTROS"

is_food_udf = udf(is_food_related, BooleanType())
get_segmento_udf = udf(get_segmento_alimentacao, StringType())

def calcular_completude(df: DataFrame, colunas_obrigatorias: list) -> dict:
    total_registros = df.count()
    metricas = {}
    for coluna in colunas_obrigatorias:
        nao_nulos = df.filter(col(coluna).isNotNull()).count()
        taxa_completude = (nao_nulos / total_registros * 100) if total_registros > 0 else 0
        metricas[coluna] = {
            'total': total_registros,
            'preenchidos': nao_nulos,
            'nulos': total_registros - nao_nulos,
            'taxa_completude_%': round(taxa_completude, 2)
        }
    return metricas

def validar_precisao_numerica(df: DataFrame, coluna: str, min_val: float = None, max_val: float = None) -> DataFrame:
    condicao = col(coluna).isNotNull()
    if min_val is not None:
        condicao = condicao & (col(coluna) >= min_val)
    if max_val is not None:
        condicao = condicao & (col(coluna) <= max_val)
    return df.filter(condicao)

def remover_duplicados(df: DataFrame, chave_primaria: list, criterio_desempate: str = "hora_ingestao") -> DataFrame:
    window_spec = Window.partitionBy(chave_primaria).orderBy(col(criterio_desempate).desc())
    df_deduplicated = df.withColumn("row_num", row_number().over(window_spec)) \
                        .filter(col("row_num") == 1) \
                        .drop("row_num")
    return df_deduplicated

def adicionar_metadados_silver(df: DataFrame) -> DataFrame:
    return df.withColumn("data_processamento_silver", 
                         from_utc_timestamp(current_timestamp(), "America/Sao_Paulo"))

# ========== ADICIONANDO CAMPO review_food_count NA silver_user ==========
#mover para os locais corretos 

#mover para célula de tratamento da user

print("ADICIONANDO campo review_food_count na silver_user")
print("="*60)

# mover para tratamento de user
# Carrega tabelas silver
df_user = spark.table(f"{CATALOG}.{SCHEMA_SILVER}.silver_user")
df_review = spark.table(f"{CATALOG}.{SCHEMA_SILVER}.silver_review")
df_business = spark.table(f"{CATALOG}.{SCHEMA_SILVER}.silver_business")

print(f"Total de usuários: {df_user.count()}")

#excluir
# Remove coluna review_food_count se já existir (para evitar ambiguidade)
if "review_food_count" in df_user.columns:
    df_user = df_user.drop("review_food_count")
    print("✓ Coluna review_food_count antiga removida")

#excluir business, nao tem outros
# Filtra food_category desejadas
food_categories = ["RESTAURANTE", "BAR E BEBIDA", "PADARIA", "CAFE"]
df_business_food = df_business.filter(col("food_category").isin(food_categories))
print(f"Estabelecimentos nas categorias desejadas: {df_business_food.count()}")

# Join review com business para pegar apenas reviews de estabelecimentos de alimentação
df_review_food = df_review.join(
    df_business_food.select("business_id", "food_category"),
    on="business_id",
    how="inner"
)
print(f"Reviews de estabelecimentos filtrados: {df_review_food.count()}")


# Calcula review_food_count por user_id
df_review_food_count = df_review_food.groupBy("user_id").agg(
    count("review_id").alias("review_food_count")
)

# LEFT JOIN: mantém todos os usuários, adiciona review_food_count (0 se não tiver reviews)
df_user_updated = df_user.join(
    df_review_food_count,
    on="user_id",
    how="left"
).withColumn(
    "review_food_count",
    when(col("review_food_count").isNull(), lit(0)).otherwise(col("review_food_count"))
)

print(f"\nTodos os usuários mantidos: {df_user_updated.count()}")

# Verifica distribuição
users_with_food_reviews = df_user_updated.filter(col("review_food_count") > 0).count()
users_without_food_reviews = df_user_updated.filter(col("review_food_count") == 0).count()

print(f"  - Usuários com reviews nas categorias: {users_with_food_reviews}")
print(f"  - Usuários sem reviews nas categorias: {users_without_food_reviews}")

# Adiciona metadados silver
df_user_silver = adicionar_metadados_silver(df_user_updated)

# Salva tabela silver_user atualizada com overwriteSchema=True
table_silver = f"{CATALOG}.{SCHEMA_SILVER}.silver_user"
df_user_silver.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(table_silver)

print(f"\n✓ Campo review_food_count adicionado à tabela silver_user")
print(f"Categorias consideradas: {food_categories}")
print(f"Total de registros: {df_user_silver.count()}")
print("="*60)

In [0]:
# ========== TABELA 1: BUSINESS ==========

table_name = "bronze_yelp_academic_dataset_business"
print(f"Processando tabela: {table_name}")
print("="*60)

# Leitura da tabela bronze
df_business = spark.table(f"{CATALOG}.{SCHEMA_BRONZE}.{table_name}")
print(f"Registros originais: {df_business.count()}")

# ========== FILTRO: ESTABELECIMENTOS DE ALIMENTAÇÃO ==========
print("\n--- Filtro de Estabelecimentos de Alimentação ---")
df_business = df_business.filter(is_food_udf(col('categories')))
print(f"Registros relacionados a alimentação: {df_business.count()}")

# Adiciona coluna de food_category
df_business = df_business.withColumn('food_category', get_segmento_udf(col('categories')))
print("✓ Coluna 'food_category' adicionada")

# Filtra para não incluir 'OUTROS' na silver
df_business = df_business.filter(col('food_category') != "OUTROS")
print("✓ Registros com food_category = 'OUTROS' removidos")

# Mostra distribuição por food_category
print("\nDistribuição por food_category:")
display(df_business.groupBy('food_category').count().orderBy(col('count').desc()))

# COMPLETUDE: Campos obrigatórios
colunas_obrigatorias = ['business_id', 'name', 'city', 'state']
metricas_completude = calcular_completude(df_business, colunas_obrigatorias)

print("\n--- Métricas de COMPLETUDE ---")
for col_name, metricas in metricas_completude.items():
    print(f"  {col_name}: {metricas['taxa_completude_%']}% completo ({metricas['nulos']} nulos)")

# Filtra registros com campos obrigatórios preenchidos
df_business_clean = df_business.filter(
    col('business_id').isNotNull() & 
    col('name').isNotNull() & 
    col('city').isNotNull() & 
    col('state').isNotNull()
)

print(f"\nApós filtro de completude: {df_business_clean.count()} registros")

# PRECISÃO: Validação de ranges numéricos
print("\n--- Validação de PRECISÃO ---")

# Stars: 0 a 5
df_business_clean = validar_precisao_numerica(df_business_clean, 'stars', min_val=0, max_val=5)
print(f"  Stars (0-5): {df_business_clean.count()} registros válidos")

# Latitude: -90 a 90
df_business_clean = validar_precisao_numerica(df_business_clean, 'latitude', min_val=-90, max_val=90)
print(f"  Latitude (-90/90): {df_business_clean.count()} registros válidos")

# Longitude: -180 a 180
df_business_clean = validar_precisao_numerica(df_business_clean, 'longitude', min_val=-180, max_val=180)
print(f"  Longitude (-180/180): {df_business_clean.count()} registros válidos")

# Review count: >= 0
df_business_clean = df_business_clean.filter(col('review_count') >= 0)
print(f"  Review count (>=0): {df_business_clean.count()} registros válidos")

# Remoção de duplicados
print("\n--- Remoção de Duplicados ---")
df_business_clean = remover_duplicados(df_business_clean, ['business_id'])
print(f"Após deduplication: {df_business_clean.count()} registros")

# ========== PADRONIZAÇÃO DO ENDEREÇO ==========
print("\n--- Padronização de Endereços ---")
from pyspark.sql.functions import upper, regexp_replace, trim

# Converte endereço para caixa alta
df_business_clean = df_business_clean.withColumn("address", upper(col("address")))
print("✓ Coluna 'address' convertida para caixa alta")

# ========== PADRONIZAÇÃO DO NOME ==========
print("\n--- Padronização de Nomes ---")

# Apply transformations step by step
name_col = upper(col("name"))
name_col = regexp_replace(name_col, r",.*", "")  # Remove text after comma
name_col = regexp_replace(name_col, r",", " ")  # Replace comma with space
name_col = regexp_replace(name_col, r"\.", " ")  # Replace dot with space
name_col = regexp_replace(name_col, r"ST\.", "SAINT")
name_col = regexp_replace(name_col, r"\bST\b", "SAINT")
name_col = regexp_replace(name_col, r"\bST\.\b", "SAINT")
name_col = regexp_replace(name_col, r"SAINTT", "SAINT")
name_col = regexp_replace(name_col, r"NW ", "NEW")
name_col = regexp_replace(name_col, r"'", " ")  # Replace apostrophes
name_col = regexp_replace(name_col, r"-", " ")  # Replace hyphens
name_col = regexp_replace(name_col, r"^ +| +$", "")  # Trim edges
name_col = regexp_replace(name_col, r" {2,}", " ")  # Collapse multiple spaces
name_col = trim(name_col)

df_business_clean = df_business_clean.withColumn("name_validated", name_col)
print("✓ Coluna 'name_validated' adicionada")

# ========== PADRONIZAÇÃO DA CIDADE ==========
print("\n--- Padronização de Cidades ---")

# Apply transformations step by step
city_col = upper(col("city"))
city_col = regexp_replace(city_col, r",.*", "")  # Remove content after comma
city_col = regexp_replace(city_col, r"/.*", "")  # Remove content after slash
city_col = regexp_replace(city_col, r"BCH", "BEACH")
city_col = regexp_replace(city_col, r",", " ")
city_col = regexp_replace(city_col, r"/", " ")
city_col = regexp_replace(city_col, r"%MTLAUREL%", "MT LAUREL")
city_col = regexp_replace(city_col, r"%TAMPA FLORIDA%", "TAMPA")
city_col = regexp_replace(city_col, r"TWP", "TOWNSHIP")
city_col = regexp_replace(city_col, r"MT \.", "MT")
city_col = regexp_replace(city_col, r"MT\.", "MT")
city_col = regexp_replace(city_col, r"SAINTLOUIS", "SAINT LOUIS")
city_col = regexp_replace(city_col, r"SAINTT", "SAINT")
city_col = regexp_replace(city_col, r"^SAINT PETE$", "SAINT PETERSBURG")
city_col = regexp_replace(city_col, r"SAINT PETERS", "SAINT PETERSBURG")
city_col = regexp_replace(city_col, r"REDINGTN", "REDINGTON")
city_col = regexp_replace(city_col, r"REDNGTN", "REDINGTON")
city_col = regexp_replace(city_col, r"CHALEMETTE", "CHALMETTE")
city_col = regexp_replace(city_col, r"INPOLIS", "INDIANAPOLIS")
city_col = regexp_replace(city_col, r"CONSHOHOEKEN", "CONSHOHOCKEN")
city_col = regexp_replace(city_col, r"FESTERVILLE", "FEASTERVILLE")
city_col = regexp_replace(city_col, r"TOWSSHIP", "TOWNSHIP")
city_col = regexp_replace(city_col, r"%TWN%", "TOWN")
city_col = regexp_replace(city_col, r"CNTRY", "COUNTRY")
city_col = regexp_replace(city_col, r"TIERRE VERDE", "TIERRA VERDE")
city_col = regexp_replace(city_col, r"NW", "NEW")
city_col = regexp_replace(city_col, r"TOWN & COUNTRY", "TOWN N COUNTRY")
city_col = regexp_replace(city_col, r"\.", " ")  # Replace dot with space
city_col = regexp_replace(city_col, r"ST\.", "SAINT")
city_col = regexp_replace(city_col, r"\bST\b", "SAINT")
city_col = regexp_replace(city_col, r"\bST\.\b", "SAINT")
city_col = regexp_replace(city_col, r"'", " ")  # Replace apostrophes
city_col = regexp_replace(city_col, r"-", " ")  # Replace hyphens
city_col = regexp_replace(city_col, r" {2,}", " ")  # Collapse multiple spaces
city_col = trim(city_col)  # Remove spaces at the beginning and end

df_business_clean = df_business_clean.withColumn("city_validated", city_col)
print("✓ Coluna 'city_validated' adicionada")

# ========== SUBSTITUIÇÃO DOS CAMPOS ORIGINAIS ==========
print("\n--- Substituição de Campos Originais ---")

# Remove campos originais name e city
df_business_clean = df_business_clean.drop('name', 'city')
print("✓ Campos originais 'name' e 'city' removidos")

# Renomeia os campos validados
df_business_clean = df_business_clean.withColumnRenamed('name_validated', 'name')
df_business_clean = df_business_clean.withColumnRenamed('city_validated', 'city')
print("✓ Campos renomeados: 'name_validated' -> 'name', 'city_validated' -> 'city'")

# Adiciona metadados silver
df_business_silver = adicionar_metadados_silver(df_business_clean)

# Remove coluna antiga 'segmento' se existir (para evitar duplicação com food_category)
if 'segmento' in df_business_silver.columns:
    df_business_silver = df_business_silver.drop('segmento')
    print("\n✓ Coluna antiga 'segmento' removida")

# Salva na camada Silver
table_silver = f"{CATALOG}.{SCHEMA_SILVER}.silver_business"
df_business_silver.write.mode("overwrite").saveAsTable(table_silver)

print(f"\n✓ Tabela silver criada: {table_silver}")
print(f"Total de registros silver: {df_business_silver.count()}")
print("="*60)

In [0]:
# Visualiza amostra dos dados limpos
print("AMOSTRA - Tabela Silver Business:")
print("="*60)

df_sample = spark.table(f"{CATALOG}.{SCHEMA_SILVER}.silver_business")

# Métricas gerais
print(f"Total de registros: {df_sample.count()}")
print(f"Campos: {len(df_sample.columns)}")

# Amostra de dados
print("\nPrimeiros 10 registros:")
display(df_sample.select(
    'business_id', 
    'name',
    'city',
    'state', 
    'stars', 
    'review_count',
    'food_category',
    'data_processamento_silver'
).limit(10))

# Estatísticas de qualidade
print("\nEstatísticas de Stars:")
df_sample.select('stars').describe().show()

print("\nDistribuição por Estado (Top 10):")
display(df_sample.groupBy('state').count().orderBy(col('count').desc()).limit(10))

In [0]:
%sql
SELECT 
  name,
  address,
  city,
  state,
  food_category
FROM workspace.yelp_ing.silver_business
WHERE address IS NOT NULL
LIMIT 10

In [0]:
# ========== AMOSTRAS DE PADRONIZAÇÃO DE NOMES ==========

print("ANÁLISE DE PADRONIZAÇÃO DE NOMES")
print("="*80)

df_business = spark.table(f"{CATALOG}.{SCHEMA_SILVER}.silver_business")

# Amostra geral
print("\n1. Amostra Geral (20 registros):")
print("-"*80)
display(df_business.select('name', 'city', 'state').limit(20))

# Estatísticas
print("\n2. Estatísticas de Transformação:")
print("-"*80)
total = df_business.count()

print(f"Total de estabelecimentos: {total:,}")
print(f"Campos 'name' e 'city' padronizados com sucesso!")

print("\n✓ Análise de padronização concluída!")

In [0]:
# ========== TABELA 2: REVIEW ==========

table_name = "bronze_yelp_academic_dataset_review"
print(f"Processando tabela: {table_name}")
print("="*60)

# Leitura da tabela bronze
df_review = spark.table(f"{CATALOG}.{SCHEMA_BRONZE}.{table_name}")
print(f"Registros originais: {df_review.count()}")

# COMPLETUDE: Campos obrigatórios
colunas_obrigatorias = ['review_id', 'user_id', 'business_id', 'stars', 'text', 'date']
metricas_completude = calcular_completude(df_review, colunas_obrigatorias)

print("\n--- Métricas de COMPLETUDE ---")
for col_name, metricas in metricas_completude.items():
    print(f"  {col_name}: {metricas['taxa_completude_%']}% completo ({metricas['nulos']} nulos)")

# Filtra registros com campos obrigatórios preenchidos
df_review_clean = df_review.filter(
    col('review_id').isNotNull() & 
    col('user_id').isNotNull() & 
    col('business_id').isNotNull() &
    col('stars').isNotNull() &
    col('text').isNotNull() &
    col('date').isNotNull()
)

print(f"\nApós filtro de completude: {df_review_clean.count()} registros")

# PRECISÃO: Validações
print("\n--- Validação de PRECISÃO ---")

# Stars: 1 a 5 (reviews não permitem 0 stars)
df_review_clean = validar_precisao_numerica(df_review_clean, 'stars', min_val=1, max_val=5)
print(f"  Stars (1-5): {df_review_clean.count()} registros válidos")

# Texto não vazio (após trim)
df_review_clean = df_review_clean.filter(length(trim(col('text'))) > 0)
print(f"  Texto não vazio: {df_review_clean.count()} registros válidos")

# Useful, funny, cool >= 0
for coluna in ['useful', 'funny', 'cool']:
    df_review_clean = df_review_clean.filter(
        col(coluna).isNull() | (col(coluna) >= 0)
    )
print(f"  Contadores (useful/funny/cool >=0): {df_review_clean.count()} registros válidos")

# Remoção de duplicados
print("\n--- Remoção de Duplicados ---")
df_review_clean = remover_duplicados(df_review_clean, ['review_id'])
print(f"Após deduplication: {df_review_clean.count()} registros")

# Adiciona metadados silver
df_review_silver = adicionar_metadados_silver(df_review_clean)

# Salva na camada Silver
table_silver = f"{CATALOG}.{SCHEMA_SILVER}.silver_review"
df_review_silver.write.mode("overwrite").saveAsTable(table_silver)

print(f"\n✓ Tabela silver criada: {table_silver}")
print(f"Total de registros silver: {df_review_silver.count()}")
print("="*60)

In [0]:
# Visualiza amostra dos dados limpos
print("AMOSTRA - Tabela Silver Review:")
print("="*60)

df_sample = spark.table(f"{CATALOG}.{SCHEMA_SILVER}.silver_review")

print(f"Total de reviews: {df_sample.count()}")

# Amostra de dados
print("\nPrimeiros 5 registros:")
display(df_sample.select(
    'review_id', 
    'user_id', 
    'business_id', 
    'stars',
    'date',
    'data_processamento_silver'
).limit(5))

# Distribuição de stars
print("\nDistribuição de Stars:")
display(df_sample.groupBy('stars').count().orderBy('stars'))

In [0]:
# ========== TABELA 3: USER ==========

CATALOG = "workspace"
SCHEMA_BRONZE = "yelp_ing"

table_name = "bronze_yelp_academic_dataset_user"
print(f"Processando tabela: {table_name}")
print("="*60)

# Leitura da tabela bronze
df_user = spark.table(f"{CATALOG}.{SCHEMA_BRONZE}.{table_name}")
print(f"Registros originais: {df_user.count()}")

# COMPLETUDE: Campos obrigatórios
colunas_obrigatorias = ['user_id', 'name', 'yelping_since']
metricas_completude = calcular_completude(df_user, colunas_obrigatorias)

print("\n--- Métricas de COMPLETUDE ---")
for col_name, metricas in metricas_completude.items():
    print(f"  {col_name}: {metricas['taxa_completude_%']}% completo ({metricas['nulos']} nulos)")

# Filtra registros com campos obrigatórios preenchidos
df_user_clean = df_user.filter(
    col('user_id').isNotNull() & 
    col('name').isNotNull() & 
    col('yelping_since').isNotNull()
)

print(f"\nApós filtro de completude: {df_user_clean.count()} registros")

# PRECISÃO: Validações numéricas
print("\n--- Validação de PRECISÃO ---")

# Average stars: 0 a 5
df_user_clean = validar_precisao_numerica(df_user_clean, 'average_stars', min_val=0, max_val=5)
print(f"  Average stars (0-5): {df_user_clean.count()} registros válidos")

# Review count, fans, useful, funny, cool >= 0
for coluna in ['review_count', 'fans', 'useful', 'funny', 'cool']:
    df_user_clean = df_user_clean.filter(
        col(coluna).isNull() | (col(coluna) >= 0)
    )
print(f"  Contadores (>=0): {df_user_clean.count()} registros válidos")

# Compliments >= 0
compliment_cols = [c for c in df_user_clean.columns if c.startswith('compliment_')]
for coluna in compliment_cols:
    df_user_clean = df_user_clean.filter(
        col(coluna).isNull() | (col(coluna) >= 0)
    )
print(f"  Compliments (>=0): {df_user_clean.count()} registros válidos")

# Remoção de duplicados
print("\n--- Remoção de Duplicados ---")
df_user_clean = remover_duplicados(df_user_clean, ['user_id'])
print(f"Após deduplication: {df_user_clean.count()} registros")

# ========== ADICIONA CAMPO review_food_count ==========
print("\n--- Adicionando campo review_food_count ---")

# Carrega tabelas review e business já processadas
df_review = spark.table(f"{CATALOG}.{SCHEMA_SILVER}.silver_review")
df_business = spark.table(f"{CATALOG}.{SCHEMA_SILVER}.silver_business")

# Filtra food_category desejadas
food_categories = ["RESTAURANTE", "BAR E BEBIDA", "PADARIA", "CAFE"]
df_business_food = df_business.filter(col("food_category").isin(food_categories))
print(f"  Estabelecimentos nas categorias food: {df_business_food.count()}")

# Join review com business para pegar apenas reviews de estabelecimentos de alimentação
df_review_food = df_review.join(
    df_business_food.select("business_id", "food_category"),
    on="business_id",
    how="inner"
)
print(f"  Reviews de estabelecimentos food: {df_review_food.count()}")

# Calcula review_food_count por user_id
df_review_food_count = df_review_food.groupBy("user_id").agg(
    count("review_id").alias("review_food_count")
)

# LEFT JOIN: adiciona review_food_count (0 se não tiver reviews)
df_user_clean = df_user_clean.join(
    df_review_food_count,
    on="user_id",
    how="left"
).withColumn(
    "review_food_count",
    when(col("review_food_count").isNull(), lit(0)).otherwise(col("review_food_count"))
)

users_with_food = df_user_clean.filter(col("review_food_count") > 0).count()
users_without_food = df_user_clean.filter(col("review_food_count") == 0).count()
print(f"  Usuários com reviews food: {users_with_food}")
print(f"  Usuários sem reviews food: {users_without_food}")

# Adiciona metadados silver
df_user_silver = adicionar_metadados_silver(df_user_clean)

# Salva na camada Silver
table_silver = f"{CATALOG}.{SCHEMA_SILVER}.silver_user"
df_user_silver.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(table_silver)

print(f"\n✓ Tabela silver criada: {table_silver}")
print(f"Total de registros silver: {df_user_silver.count()}")
print(f"Campo review_food_count incluído (baseado em food_category)")
print("="*60)

In [0]:
# ========== TABELA 4: TIP ==========

table_name = "bronze_yelp_academic_dataset_tip"
print(f"Processando tabela: {table_name}")
print("="*60)

# Leitura da tabela bronze
df_tip = spark.table(f"{CATALOG}.{SCHEMA_BRONZE}.{table_name}")
print(f"Registros originais: {df_tip.count()}")

# COMPLETUDE: Campos obrigatórios
colunas_obrigatorias = ['user_id', 'business_id', 'text', 'date']
metricas_completude = calcular_completude(df_tip, colunas_obrigatorias)

print("\n--- Métricas de COMPLETUDE ---")
for col_name, metricas in metricas_completude.items():
    print(f"  {col_name}: {metricas['taxa_completude_%']}% completo ({metricas['nulos']} nulos)")

# Filtra registros com campos obrigatórios preenchidos
df_tip_clean = df_tip.filter(
    col('user_id').isNotNull() & 
    col('business_id').isNotNull() & 
    col('text').isNotNull() &
    col('date').isNotNull()
)

print(f"\nApós filtro de completude: {df_tip_clean.count()} registros")

# PRECISÃO: Validações
print("\n--- Validação de PRECISÃO ---")

# Texto não vazio
df_tip_clean = df_tip_clean.filter(length(trim(col('text'))) > 0)
print(f"  Texto não vazio: {df_tip_clean.count()} registros válidos")

# Compliment count >= 0
df_tip_clean = df_tip_clean.filter(
    col('compliment_count').isNull() | (col('compliment_count') >= 0)
)
print(f"  Compliment count (>=0): {df_tip_clean.count()} registros válidos")

# Remoção de duplicados (chave composta: user_id + business_id + date + text)
print("\n--- Remoção de Duplicados ---")
df_tip_clean = remover_duplicados(df_tip_clean, ['user_id', 'business_id', 'date', 'text'])
print(f"Após deduplication: {df_tip_clean.count()} registros")

# Adiciona metadados silver
df_tip_silver = adicionar_metadados_silver(df_tip_clean)

# Salva na camada Silver
table_silver = f"{CATALOG}.{SCHEMA_SILVER}.silver_tip"
df_tip_silver.write.mode("overwrite").saveAsTable(table_silver)

print(f"\n✓ Tabela silver criada: {table_silver}")
print(f"Total de registros silver: {df_tip_silver.count()}")
print("="*60)

In [0]:
# ========== TABELA 5: CHECKIN ==========

table_name = "bronze_yelp_academic_dataset_checkin"
print(f"Processando tabela: {table_name}")
print("="*60)

# Leitura da tabela bronze
df_checkin = spark.table(f"{CATALOG}.{SCHEMA_BRONZE}.{table_name}")
print(f"Registros originais: {df_checkin.count()}")

# COMPLETUDE: Campos obrigatórios
colunas_obrigatorias = ['business_id', 'date']
metricas_completude = calcular_completude(df_checkin, colunas_obrigatorias)

print("\n--- Métricas de COMPLETUDE ---")
for col_name, metricas in metricas_completude.items():
    print(f"  {col_name}: {metricas['taxa_completude_%']}% completo ({metricas['nulos']} nulos)")

# Filtra registros com campos obrigatórios preenchidos
df_checkin_clean = df_checkin.filter(
    col('business_id').isNotNull() & 
    col('date').isNotNull()
)

print(f"\nApós filtro de completude: {df_checkin_clean.count()} registros")

# PRECISÃO: Validações
print("\n--- Validação de PRECISÃO ---")

# Date não vazio (campo contém timestamps separados por vírgula)
df_checkin_clean = df_checkin_clean.filter(length(trim(col('date'))) > 0)
print(f"  Date não vazio: {df_checkin_clean.count()} registros válidos")

# Remoção de duplicados
print("\n--- Remoção de Duplicados ---")
df_checkin_clean = remover_duplicados(df_checkin_clean, ['business_id'])
print(f"Após deduplication: {df_checkin_clean.count()} registros")

# Adiciona metadados silver
df_checkin_silver = adicionar_metadados_silver(df_checkin_clean)

# Salva na camada Silver
table_silver = f"{CATALOG}.{SCHEMA_SILVER}.silver_checkin"
df_checkin_silver.write.mode("overwrite").saveAsTable(table_silver)

print(f"\n✓ Tabela silver criada: {table_silver}")
print(f"Total de registros silver: {df_checkin_silver.count()}")
print("="*60)

In [0]:
# ========== RESUMO FINAL: QUALIDADE DE DADOS SILVER ==========

print("RELATÓRIO DE QUALIDADE - CAMADA SILVER")
print("="*80)
print("\nDimensões de Qualidade Aplicadas:")
print("  1. COMPLETUDE: Campos obrigatórios preenchidos")
print("  2. PRECISÃO: Valores dentro dos ranges esperados\n")
print("="*80)

# Lista de tabelas silver
tabelas_silver = [
    'silver_business',
    'silver_review',
    'silver_user',
    'silver_tip',
    'silver_checkin'
]

resumo = []

for tabela in tabelas_silver:
    try:
        df = spark.table(f"{CATALOG}.{SCHEMA_SILVER}.{tabela}")
        count = df.count()
        colunas = len(df.columns)
        
        # Verifica se tem o campo de processamento
        tem_metadata = 'data_processamento_silver' in df.columns
        
        resumo.append({
            'Tabela': tabela,
            'Registros': count,
            'Colunas': colunas,
            'Metadata Silver': '✓' if tem_metadata else '✗'
        })
        
    except Exception as e:
        print(f"Erro ao processar {tabela}: {str(e)}")

# Cria DataFrame de resumo
import pandas as pd
df_resumo = pd.DataFrame(resumo)

print("\nTABELAS SILVER CRIADAS:")
print(df_resumo.to_string(index=False))

print("\n" + "="*80)
print("\nPRÓXIMOS PASSOS RECOMENDADOS:")
print("  1. Validar relacionamentos entre tabelas (FKs)")
print("  2. Criar testes de qualidade automatizados")
print("  3. Implementar monitoramento contínuo")
print("  4. Documentar regras de negócio aplicadas")
print("  5. Criar camada Gold com agregações analíticas")
print("\n" + "="*80)
print("\n✓ Processo de limpeza concluído com sucesso!")